In [1]:
# 7. Embedding-grounded versus zero-shot prompt

import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd

DATA = Path("../data")   # not included in the repository
TAB = Path("../tab")

RUN_DIR = DATA / "llm" / "zero_shot"
CONDITIONS = ["few_shot", "zero_shot"]
SEED = 20261105

In [2]:
# Input

with open(DATA / "llm" / "geluid_annotaties_predictions.json") as f:
    tasks = [t["data"] for t in json.load(f)]

def task_id(meta):
    return f"{meta['source_file'].removesuffix('_final.xml')}_p{meta['page']}_i{meta['incident']}"

incidents = pd.DataFrame({"id": [task_id(t["meta"]) for t in tasks], "text": [t["text"] for t in tasks]})
dev_ids = set((RUN_DIR / "dev_ids.txt").read_text().split())
test = incidents[~incidents["id"].isin(dev_ids)].reset_index(drop=True)
texts = dict(zip(test["id"], test["text"]))

print(f"annotated: {len(incidents):,} | development: {len(dev_ids)} | test: {len(test):,}")
assert test.equals(pd.read_csv(RUN_DIR / "incidents_d3_nocturnal.csv", keep_default_na=False))

annotated: 15,683 | development: 200 | test: 15,483


In [3]:
# Model runs

RUN = False

if RUN:
    import time
    import anthropic

    client = anthropic.Anthropic()
    for condition in CONDITIONS:
        prompt = open(f"prompts/{condition}.txt").read()
        requests = [{"custom_id": f"r{i}",
                     "params": {"model": "claude-haiku-4-5-20251001", "max_tokens": 1000, "system": prompt,
                                "messages": [{"role": "user", "content": text}]}}
                    for i, text in enumerate(test["text"])]
        batch = client.messages.batches.create(requests=requests).id
        while client.messages.batches.retrieve(batch).processing_status != "ended":
            time.sleep(60)
        with open(RUN_DIR / f"{condition}.jsonl", "w") as out:
            for res in client.messages.batches.results(batch):
                record = {"id": test["id"][int(res.custom_id[1:])], "status": res.result.type}
                if res.result.type == "succeeded":
                    record["raw"] = res.result.message.content[0].text
                out.write(json.dumps(record, ensure_ascii=False) + "\n")

In [4]:
# Parsing the responses

FENCED = re.compile(r"```(?:json)?\s*(.*?)```", re.S)
FENCE = re.compile(r"^```(?:json)?\s*|\s*```$")
FEW_SHOT_LABELS = {"singing", "yelling", "doorbell_ringing", "playing_inn_music", "playing_street_music",
                   "scolding", "disrupting_nightrest", "traffic_noise", "making_noise", "other_sound"}

def read_json(text):
    """Parse a model response. If the model revised itself in several code blocks, the last valid one counts."""
    blocks = FENCED.findall(text)
    for block in reversed(blocks if len(blocks) > 1 else []):
        try:
            return json.loads(block)
        except json.JSONDecodeError:
            pass
    s = FENCE.sub("", text.strip()).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        start = min(i for i in (s.find("["), s.find("{")) if i != -1)
        return json.loads(s[start:max(s.rfind("]"), s.rfind("}")) + 1])

def noise_complaint(condition, obj):
    if condition == "few_shot":
        labels = {str(span["label"]).strip().lower().replace(" ", "_") for span in obj}
        if not labels <= FEW_SHOT_LABELS:
            raise ValueError(f"unknown label: {labels - FEW_SHOT_LABELS}")
        return bool(labels & {"making_noise", "disrupting_nightrest"})
    if not isinstance(obj.get("noise_complaint"), bool):
        raise ValueError("no noise_complaint field")
    return obj["noise_complaint"]

predicted, flagged = {}, set()
for condition in CONDITIONS:
    predicted[condition], failed = {}, []
    for line in open(RUN_DIR / f"{condition}.jsonl"):
        record = json.loads(line)
        try:
            obj = read_json(record["raw"])
            predicted[condition][record["id"]] = noise_complaint(condition, obj)
            if obj and (condition == "few_shot" or obj["spans"] or obj["noise_complaint"]):
                flagged.add(record["id"])
        except Exception:
            predicted[condition][record["id"]] = False
            flagged.add(record["id"])
            failed.append(record["id"])
    print(f"{condition}: {len(predicted[condition]):,} responses, {sum(predicted[condition].values()):,} "
          f"noise complaints, unparsable: {failed}")

few_shot: 15,483 responses, 996 noise complaints, unparsable: []
zero_shot: 15,483 responses, 1,232 noise complaints, unparsable: ['MA#17871_p31_i7']


In [5]:
# Stratified sample

strata = {"flagged": sorted(i for i in test["id"] if i in flagged),
          "rest": sorted(i for i in test["id"] if i not in flagged)}
sizes = {"flagged": 200, "rest": 100}

rng = random.Random(SEED)
sample = pd.DataFrame([{"id": i, "stratum": s, "weight": len(ids) / sizes[s]}
                       for s, ids in strata.items() for i in rng.sample(ids, sizes[s])])
print({s: len(ids) for s, ids in strata.items()})
assert sample["id"].tolist() == pd.read_csv(RUN_DIR / "sample_300.csv", keep_default_na=False)["id"].tolist()

{'flagged': 3117, 'rest': 12366}


In [6]:
# Blind annotation

blind = {}
for task in json.load(open(RUN_DIR / "blind300_export.json")):
    annotation = max((a for a in task["annotations"] if not a.get("was_cancelled")),
                     key=lambda a: a.get("updated_at") or a.get("created_at") or "")
    for result in annotation["result"]:
        if result["from_name"] == "noise_complaint":
            blind[task["data"]["id"]] = result["value"]["choices"][0] == "lawaaiklacht"

corrections = pd.read_csv(RUN_DIR / "noise_corrections.csv")
corrected = blind | dict(zip(corrections["id"], corrections["noise_complaint"].astype(bool)))
print(f"noise complaints: blind {sum(blind.values())}, corrected {sum(corrected.values())}")
corrections

noise complaints: blind 41, corrected 51


,id,noise_complaint,reden
0,731#1602_p9_i3,True,"""de nachtrust storde der inwoonders"""
1,731#1608_p102_i5,True,"""nachtrust stoorde"""
2,731#1610_p279_i5,True,"""groot lawijd"""
3,731#1611_p245_i4,True,"""de nachtrust stoorde"""
4,731#1612_p556_i6,True,"""lawijt maakte"""
5,731#1613_p109_i3,True,"""de rust stoorde … met lewijt te maken"""
6,731#1594_p218_i2,True,"""nacht ramoer te maken"", ""nachtrumder"""
7,MA#17633_p80_i2,True,"""nachtrumoer maakte, van aard de rust der inwo..."
8,MA#17875_p314_i1,True,"""hevig lowaaid maakten … de nachtrust der inwo..."
9,MA#17881_p129_i2,True,"""nachtlawijd"""


In [7]:
# Zero-shot comparison

weight = dict(zip(sample["id"], sample["weight"]))
by_stratum = {s: sample.loc[sample["stratum"] == s, "id"].tolist() for s in sizes}

def scores(ids, gold, condition):
    tp = sum(weight[i] for i in ids if gold[i] and predicted[condition][i])
    fp = sum(weight[i] for i in ids if not gold[i] and predicted[condition][i])
    fn = sum(weight[i] for i in ids if gold[i] and not predicted[condition][i])
    p, r = tp / (tp + fp), tp / (tp + fn)
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r)}

def interval(values):
    v = sorted(values)
    return v[int(0.025 * (len(v) - 1))], v[int(0.975 * (len(v) - 1))]

rng = random.Random(SEED)
boots = [[i for ids in by_stratum.values() for i in rng.choices(ids, k=len(ids))] for _ in range(2000)]

rows, ids = [], sample["id"].tolist()
for version, gold in [("corrected", corrected), ("blind", blind)]:
    point = {c: scores(ids, gold, c) for c in CONDITIONS}
    boot = {c: [scores(b, gold, c) for b in boots] for c in CONDITIONS}
    for c in CONDITIONS:
        n = Counter((predicted[c][i], gold[i]) for i in ids)
        row = {"gold": version, "prompt": c, "tp": n[True, True], "fp": n[True, False], "fn": n[False, True],
               "estimated_complaints": round(sum(weight[i] for i in ids if gold[i]))}
        for m in ["precision", "recall", "f1"]:
            row[m] = point[c][m]
            row[f"{m}_low"], row[f"{m}_high"] = interval([b[m] for b in boot[c]])
        rows.append(row)
    row = {"gold": version, "prompt": "zero-shot minus few-shot"}
    for m in ["precision", "recall", "f1"]:
        row[m] = point["zero_shot"][m] - point["few_shot"][m]
        row[f"{m}_low"], row[f"{m}_high"] = interval([z[m] - f[m] for f, z in zip(boot["few_shot"], boot["zero_shot"])])
    rows.append(row)

zero_shot = pd.DataFrame(rows)
zero_shot.round(3).to_csv(TAB / "zero_shot_comparison.csv", index=False)
zero_shot.round(2)

,gold,prompt,tp,fp,fn,estimated_complaints,precision,precision_low,precision_high,recall,recall_low,recall_high,f1,f1_low,f1_high
0,corrected,few_shot,49.0,20.0,2.0,795.0,0.71,0.60,0.81,0.96,0.90,1.00,0.82,0.73,0.89
1,corrected,zero_shot,47.0,37.0,4.0,795.0,0.56,0.45,0.66,0.92,0.84,0.98,0.70,0.60,0.78
2,corrected,zero-shot minus few-shot,NaN,NaN,NaN,NaN,-0.15,-0.25,-0.05,-0.04,-0.12,0.04,-0.12,-0.20,-0.04
3,blind,few_shot,37.0,32.0,4.0,639.0,0.54,0.41,0.65,0.90,0.80,0.98,0.67,0.56,0.76
4,blind,zero_shot,36.0,48.0,5.0,639.0,0.43,0.33,0.53,0.88,0.77,0.97,0.58,0.47,0.67
5,blind,zero-shot minus few-shot,NaN,NaN,NaN,NaN,-0.11,-0.19,-0.02,-0.02,-0.13,0.08,-0.10,-0.18,-0.02


In [8]:
# Errors (corrected gold standard)

for c in CONDITIONS:
    fp = sorted(i for i in ids if predicted[c][i] and not corrected[i])
    fn = sorted(i for i in ids if corrected[i] and not predicted[c][i])
    print(f"{c}: {len(fp)} false positives, {len(fn)} missed")
    for i in fn:
        print(f"  missed {i}: {texts[i][:120]}")

few_shot: 20 false positives, 2 missed
  missed MA#17624_p69_i3: Mr Vergels D Heden nacht terwijl ik dienst deed aan het Kon . Taleis zag ik om 410 u dat een persoon terken deed al roep
  missed MA#17871_p31_i7: Tharse A . Duvas . KWeyler Sseanebut Heden nacht rond 2 1 / 4 ten bureele gebracht de gen SmitPetrus Janbus geb te Amste
zero_shot: 37 false positives, 4 missed
  missed 731#1613_p280_i2: id D A Heden nacht omtrent 4 ure , werd ik ver - wittigd door eenen inwooner van het huis n 6 , der Bontemantelstraat da
  missed MA#17621_p230_i2: D 2 Mr De Brael Heden nacht om 123 / 4 ten bureele gebracht de genaam de Argijrio Evang llarkopoulos geboren te Athenes 
  missed MA#17644_p68_i4: Deckers P . V . 3301 Heden avond om 22 u werd ik geroepen door Van Stitoel , Chebot geb 11 Maart 1901 te St Joos tn Mord
  missed MA#17871_p31_i7: Tharse A . Duvas . KWeyler Sseanebut Heden nacht rond 2 1 / 4 ten bureele gebracht de gen SmitPetrus Janbus geb te Amste
